In [1]:
import CoolProp.CoolProp as Cp
import math as mt
import matplotlib.pyplot as plt

def adiabiatichead(b,c,d,e,f):
    g = b*c*d*e*((f**(1/e))-1)
    return g
def compressibility(b,c,d):
    a = Cp.PropsSI('Z','P',b,'T',c,d)
    return a
def density(b,c,d):
    a = Cp.PropsSI('D','P',b,'T',c,d)
    return a
def constantpressure(b,c,d):
    a = Cp.PropsSI('C','P',b,'T',c,d)
    return a
def cpovercv(b,c,d):
    a = Cp.PropsSI('ISENTROPIC_EXPANSION_COEFFICIENT','P',b,'T',c,d)
    return a

In [2]:
################################################## Input ##########################################################

ppmm = 100        # Water concentration in the hydrogen stream, ppm
P = 19.92*(10**5) # Inlet pressure of the adsorber, bar
T = 114           # Inlet Temoerature of the adsorber, K
R = 8.314         # Gas constant, J / mol·K
M_h2o = 18        # molecular weight of water, g/mol
m = 5208          # Mass flow, kg/h

################################################# Fluid ########################################
Fluid = 'Hydrogen'
# viscosty of hydrogen
mu = Cp.PropsSI('V','P',P,'T',T,Fluid)
print('Viscosity of hydrogen',round(mu,8),'Pa·s')

rho_g = density(P,T,Fluid)
q_g = m/rho_g

################################################## Nominal operating condition #################################

P_n = 1.01325*(10**5) # bar
T_n = 273.15    # K
rho_n = density(P_n,T_n,Fluid)
q_n = m/rho_n   # Nominal Flow rate, Nm3/h

################################################## Standard operating condition ################################

P_std = 1.01325*(10**5) # bar
T_std = 288.15    # K
rho_std = density(P_std,T_std,Fluid)
q_std = m/rho_std   # Nominal Flow rate, Nm3/h

################################################## operating time and steps #############################

n_step = 2
n_hrs = 12
n_life = 5

Viscosity of hydrogen 4.63e-06 Pa·s


$$
C=\frac{{n_{gas}*MW}}{V}
$$
$$
C=\frac{{ppm*MW*P}}{R*T}
$$

In [3]:
C=(ppmm*M_h2o*P)/(R*T)
C=C/10**6
print('Actual concentration =',round(C,3),'g/m3')

Actual concentration = 3.783 g/m3


In [4]:
## standard water quantity
C_std = (ppmm*M_h2o*P_std)/(R*T_std)

print("C_stand =",round(C_std,3),'gH2O/10^6 m3')
print('C_std = ',round(C_std/10**3,6),'KgH2O/10^6 m3')

C_stand = 76130.814 gH2O/10^6 m3
C_std =  76.130814 KgH2O/10^6 m3


In [5]:
############################################## Equillibrium loading #############################################
Xe = 24 # %, Assumption
Xr = 4 # %,  Assumption
delta_X = Xe-Xr
print("Net Equillibrium Loading =",delta_X)

Net Equillibrium Loading = 20


In [6]:
############################################# Cycle during Lifetime Calculation ##################################

cycle_pertower = (1/n_hrs)*(1/n_step)*24*365*0.95*n_life
print("Number of cycles per tower =",round(cycle_pertower,0))

Number of cycles per tower = 1734.0


In [7]:
############################################ Average loading Calculation ############################################
F_l = 0.55 # based on average performance
delta_Xa = F_l*delta_X
print("Average Equillibrium Loading =",delta_Xa)

Average Equillibrium Loading = 11.0


$$
{v_{g\max }} = \frac{A}{{\sqrt {{\rho _g}} }}
$$

In [8]:
########################################### Maximum Velocity through the column ##########################################
v_gmax = 74/mt.sqrt(rho_g)
print('Maximum velocity',round(v_gmax,2),'m/min')

Maximum velocity 36.04 m/min


$$
{D_m} = \sqrt {\frac{{4{q_g}}}{{\pi {v_g}}}} 
$$

In [9]:
##################################### Minimum Bed Diameter #######################################################
D = mt.sqrt((4*q_g/60)/(mt.pi*v_gmax))
print('Minimum bed Diameter',round(D,2),'m')

Minimum bed Diameter 0.85 m


In [10]:
####################################### we can change the dimater base don our output #################################
# standard diameter
D = 1 #m

# actual superficial velocity through the column
v_g = (4*q_g/60)/(mt.pi*(D**2))

print('Actual Superficial Velocity',round(v_g,2),'m/min')

Actual Superficial Velocity 26.22 m/min


In [11]:
############################ Mass Flow rate of water removed per hour #########################

m_h2o = (C_std*q_std)/(10**9)
print('Mass Flow rate of water adsorbed =',round(m_h2o,2),'kg/hr')

Mass Flow rate of water adsorbed = 4.65 kg/hr


In [12]:
## mass of the adsorptant
me = m_h2o*n_hrs*100/delta_Xa
print('Mass of the adsorbent =',round(me,2),'kg')

Mass of the adsorbent = 507.64 kg


$$
{m_{MTZ}} = {\left( {{v_g}/A} \right)^{0.3}}{K_{MS}}\left[ {\frac{{\pi {D^2}}}{4}} \right]\frac{{{\rho _{ms}}}}{{{F_l}}}
$$

In [13]:
# Density of the adsorbent

rho_ms = 705 #kg/m3

# Aged mass transfer zone length

m_mtz = ((v_g/10.7)**0.3)*0.52*(3.14*(D**2)/4)*(rho_ms/F_l)

print("Aged mass transfer zone length",round(m_mtz,2))

Aged mass transfer zone length 684.64


In [14]:
## Total mass of sleve
m_t = me+(0.5*m_mtz)
print('total mass of desicant =',round(m_t,2),'kg')

total mass of desicant = 849.96 kg


In [15]:
# Usefull Capacity of molecular sieve
X_useful = (delta_Xa*me)/m_t
print('Usefull Capacity of molecular sieve',round(X_useful,2),'%')

Usefull Capacity of molecular sieve 6.57 %


In [16]:
## to check cycle time assumption

theta_bt = (X_useful*m_t)/(m_h2o*100)

print('Time for water breakthrough',round(theta_bt,2),'hr')

Time for water breakthrough 12.0 hr


In [17]:
# Xuseful new

X_usenew = (delta_X*me)/m_t
print('useful loading of the new desicant',round(X_usenew,2),'%')

useful loading of the new desicant 11.95 %


In [18]:
# breakthrough for new desicant
theta_bt_new = (delta_X*m_t)/(m_h2o*100)
print('Breakthrough of the new desicant',round(theta_bt_new,2),'hr')

Breakthrough of the new desicant 36.53 hr


## Calculating the height

$$
{h_B} = \frac{{{m_T}}}{{{\rho _{MS}}\frac{\pi }{4}{D^2}}}
$$

In [19]:
h_b = m_t*4/(rho_ms*3.14*D**2)
print('Height of the column =',round(h_b,2),'m')

Height of the column = 1.54 m


## Pressure drop
$$
\frac{{\Delta P}}{L} = B\mu {v_g} + C{\rho _g}v_g^2
$$

In [20]:
B = 17.7
C = 0.00319

delta_P_l = (B*mu*v_g)+(C*rho_g*v_g**2)
delta_p = delta_P_l*h_b

print('The estimated pressure drop is',round(delta_p,2),'kPa')

The estimated pressure drop is 14.2 kPa


In [21]:
## adsorber height
h_v = h_b+1.5
print('Total height of the Adsorber Column',round(h_v,2),'m')

Total height of the Adsorber Column 3.04 m
